In [ ]:
import cv2
import matplotlib.pyplot as plt

def draw_bbox(image, box, color=(255, 0, 0), thickness=2):
    x1, y1, x2, y2 = box
    return cv2.rectangle(image.copy(), (x1, y1), (x2, y2), color, thickness)

def replace_phone(image1, box1, image2, box2):
    phone_image2 = image2[box2[1]:box2[3], box2[0]:box2[2]]
    h1, w1 = box1[3] - box1[1], box1[2] - box1[0]
    phone_image2_resized = cv2.resize(phone_image2, (w1, h1))
    image1[box1[1]:box1[3], box1[0]:box1[2]] = phone_image2_resized
    
    return image1

image1 = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/july/clients_equalized_jpg_xml/61684_2020-05-18_17825_.jpg"
image2 = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/july/clients_equalized_jpg_xml/105558_2020-07-28_12449_.jpg"
image1 = cv2.imread(image1)
image2 = cv2.imread(image2)
box1 = (78,44,209,111)
box2 = (221,104,254,203)

image1_with_bbox = draw_bbox(image1, box1)
image2_with_bbox = draw_bbox(image2, box2)

result_image = replace_phone(image1, box1, image2, box2)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(image1_with_bbox, cv2.COLOR_BGR2RGB))
plt.title("Image 1 with Bounding Box")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(cv2.cvtColor(image2_with_bbox, cv2.COLOR_BGR2RGB))
plt.title("Image 2 with Bounding Box")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.title("Result Image")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import cv2
import numpy as np

def replace_phone_preserve_aspect(image1, box1, image2, box2):
    phone_image2 = image2[box2[1]:box2[3], box2[0]:box2[2]]
    h2, w2 = phone_image2.shape[:2]
    h1, w1 = box1[3] - box1[1], box1[2] - box1[0]
    ar2 = h2 / w2
    ar1 = h1 / w1
    if ar2 > ar1:
        new_h, new_w = h1, int(h1 / ar2)
    else:
        new_h, new_w = int(w1 * ar2), w1

    phone_image2_resized = cv2.resize(phone_image2, (new_w, new_h))

    phone_placeholder = np.zeros((h1, w1, 3), dtype=np.uint8)

    y_offset = (h1 - new_h) // 2
    x_offset = (w1 - new_w) // 2
    phone_placeholder[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = phone_image2_resized

    image1[box1[1]:box1[3], box1[0]:box1[2]] = phone_placeholder

    return image1

image2 = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/july/clients_equalized_jpg_xml/61684_2020-05-18_17825_.jpg"
image1 = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/july/clients_equalized_jpg_xml/105558_2020-07-28_12449_.jpg"
image1 = cv2.imread(image1)
image2 = cv2.imread(image2)

box2 = (78,44,209,111)
box1 = (221,104,254,203)

# image1 = "/home/ajeet/codework/datasets/All_cts_cellphone_dataset/3313.jpg"
# image2 = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/mobile_output/g_right_12316.jpg"
# image1 = cv2.imread(image1)
# image2 = cv2.imread(image2)

# box1 = (240,122,269,152)
# box2 = (281,85,320,139)

result_image = replace_phone_preserve_aspect(image1, box1, image2, box2)

cv2.imshow("Result", result_image)
cv2.imwrite("output_image.jpg", result_image)
cv2.waitKey(10000)
cv2.destroyAllWindows()


In [ ]:
from PIL import Image, ImageDraw
import random

def get_valid_phone_position(image, phone_width, phone_height, excluded_regions):
    img_width, img_height = image.size
    max_attempts = 100 
    for _ in range(max_attempts):
        xmin = random.randint(0, img_width - phone_width)
        ymin = random.randint(0, img_height - phone_height)
        xmax = xmin + phone_width
        ymax = ymin + phone_height

        overlaps = False
        for region in excluded_regions:
            rxmin, rymin, rxmax, rymax = region
            if not (xmax <= rxmin or xmin >= rxmax or ymax <= rymin or ymin >= rymax):
                overlaps = True
                break
        
        if not overlaps:
            return (xmin, ymin, xmax, ymax)
    
    raise ValueError("Could not find a valid placement for the phone.")

def place_phone_in_image(image1_path, image2_path, phone_coords, excluded_regions):
    image1 = Image.open(image1_path).convert("RGB")
    image2 = Image.open(image2_path).convert("RGB")
    xmin, ymin, xmax, ymax = phone_coords
    phone_crop = image2.crop((xmin, ymin, xmax, ymax))

    phone_width, phone_height = phone_crop.size

    try:
        target_coords = get_valid_phone_position(image1, phone_width, phone_height, excluded_regions)
    except ValueError:
        print("Could not find a valid position to place the phone.")
        return image1

    image1.paste(phone_crop, (target_coords[0], target_coords[1]))

    return image1


image1_path = "/home/ajeet/codework/datasets/All_cts_cellphone_dataset/23879.jpg"
image2_path = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/mobile_output/g_right_12316.jpg"

phone_coords = (281,85,320,139)

excluded_regions = [
    (10, 1, 254, 180),
    (80, 1, 120, 53), 
    # (300, 200, 350, 250),
]

output_image = place_phone_in_image(image1_path, image2_path, phone_coords, excluded_regions)
output_image.show()
output_image.save("output_with_phone.jpg")
